# 02 - Linear 层 & ReLU 激活函数

## 目标
- 手写 Linear 层：forward 和 backward
- 手写 ReLU：forward 和 backward(sigmoid有历史意义，但现在很少用)
- 理解权重矩阵 shape 变化：`(batch, 784) → (batch, 128)`
- 权重初始化：不能全 0（对称性问题）

## 核心公式

### forward
$$
z_1 = xW_1 + b_1
$$
$$
a_1 = ReLU(z_1)
$$

### backward
$$
\frac{\partial L}{\partial W} = x^T \cdot d_{out}
$$
$$
\frac{\partial L}{\partial x} = d_{out} \cdot W^T
$$
$$
\frac{\partial L}{\partial z} = d_{out} \cdot \mathbb{1}[z > 0] \quad \text{(ReLU)}
$$

In [1]:
%run 01_read_data.ipynb

In [2]:
"""
forward(也叫affine层)时候w不能全零，否则梯度一样，神经元算出来的值也一样，b可以全零
backward不直接修改参数，而是算出梯度，告诉你往哪个方向修改参数，修改多少
真正修改w的是SGD优化器

防止梯度消失和梯度爆炸，都希望W的方差在1附近，所以需要一个好的初始化方案（transformer不太在意了，有LayerNorm）
用方差能描述“典型大小”，矩阵的方差就是展成向量来计算
在forward阶段W的初始化中，sigmoid和tanh激活函数适合使用Xavier初始化，ReLU激活函数适合使用He初始化（因为非负砍了一半）
现在sigmoid和tanh基本不用了，但Xavier在后面也有应用

784->128->10对应x->z->y_hat(预测概率)，而Loss就是y_hat和y的差距，backward就是求这个差距对z的梯度，再对W和b的梯度，最后更新参数
x往往是(batch_size, in_features),W是(in_features, out_features)，所以一般写x @ W
"""


class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)    #这个就是He初始化
        self.b = np.zeros(out_features)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):       # dout是loss对当前层输出的梯度，dout = ∂L/∂z，就是链式法则的中间偏导
        self.dW = self.x.T @ dout
        self.db = dout.sum(axis=0)  # bias是一个神经元一个bias，所以是按列求和
        return dout @ self.W.T
    

class ReLU:
    def forward(self, x):
        self.mask = (x > 0)
        return np.maximum(0, x)

    def backward(self, dout):
        return dout * self.mask     # 会自动替换0/1
    






In [3]:
# 测试 Linear + ReLU 的 shape
linear = Linear(784, 128)
relu = ReLU()

# 取一小批数据跑 forward
batch = x_train[:3]           # (3, 784)
z1 = linear.forward(batch)    # (3, 128)
a1 = relu.forward(z1)         # (3, 128)

print(f"batch: {batch.shape}")
print(f"z1:    {z1.shape}")
print(f"a1:    {a1.shape}")

# 跑 backward 看梯度 shape 是否匹配
dout = np.random.randn(*a1.shape)  # 假装上层传来的梯度
da1 = relu.backward(dout)          # (3, 128)
dz1 = linear.backward(da1)         # (3, 784)

print(f"dout  → linear.backward: {dz1.shape}")
print(f"dW: {linear.dW.shape}, db: {linear.db.shape}")

batch: (3, 784)
z1:    (3, 128)
a1:    (3, 128)
dout  → linear.backward: (3, 784)
dW: (784, 128), db: (128,)
